[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/saly.ipynb)

# Alokace operačních sálů — vaše úloha

Notebook k bloku 5 druhého cvičení. Pracuje se **samostatně**.
Doplňte **čtyři místa `# TODO`**: data
do matice, model, kontrolu a čtení stínových cen. Volání solveru, přepočet
s hodinou navíc a graf jsou připravené; závěrem napište své rozhodnutí o nabídce.

## Zadání slovy

> Ortopedické oddělení plánuje týden. Endoprotéza kyčle spotřebuje **2,0 hodiny**
> na sále a **5 lůžkodnů**, endoprotéza kolene **2,5 hodiny** a **4 lůžkodny**.
> Marže je **62 tis. Kč** za kyčel a **55 tis. Kč** za koleno. Týdně je
> k dispozici **40 sálových hodin** a **90 lůžkodnů**.
>
> **Kolik kterých výkonů týdně plánovat, aby byla marže co největší?**

## Formulace

$$
\begin{aligned}
\text{maximize}_{\mathbf x}\quad & 62x_1 + 55x_2 && \text{marze, tis. Kc}\\
\text{subject to}\quad & 2{,}0x_1 + 2{,}5x_2 \le 40 && \text{salove hodiny}\\
& 5x_1 + 4x_2 \le 90 && \text{luzkodny}\\
& \mathbf x \ge \mathbf 0 && \text{pocty nejsou zaporne}
\end{aligned}
$$

Maticově je to $\max\ \mathbf c^\mathsf T\mathbf x$ za $A\mathbf x \le \mathbf b$,
$\mathbf x \ge \mathbf 0$. Proti infuzi z tabule je to **zrcadlový obraz**:
maximalizace místo minimalizace, $\le$ místo $\ge$ a přípustná oblast je omezená.

## Postup

1. sestavit $A$, $\mathbf b$ a $\mathbf c$ *(díra 1)*;
2. zapsat úlohu v CVXPY a vyřešit ji *(díra 2)*;
3. zkontrolovat řešení dosazením zpět *(díra 3)*;
4. přečíst stínové ceny *(díra 4)*;
5. **rozhodnout o nabídce hodiny navíc za 8 tis. Kč** jednou větou.

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/uloha_saly.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/uloha_saly.py). Notebook sám staví [`kod/postav_notebooky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/postav_notebooky.py) — ruční úprava `.ipynb` se při dalším spuštění přepíše.*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

## Data úlohy

Zadání přepsané do pojmenovaných proměnných. Nic víc — pořadí a uspořádání do
matice je vaše práce.

In [ ]:
HODIN_KYCEL, HODIN_KOLENO = 2.0, 2.5          # sálové hodiny na jeden výkon
LUZKODNU_KYCEL, LUZKODNU_KOLENO = 5.0, 4.0    # lůžkodny na jeden výkon
MARZE_KYCEL, MARZE_KOLENO = 62.0, 55.0        # marže [tis. Kč] za jeden výkon
KAPACITA_HODIN = 40.0                         # sálových hodin za týden
KAPACITA_LUZKODNU = 90.0                      # lůžkodnů za týden

## Krok 1 — matice $A$, pravá strana $\mathbf b$ a vektor marží $\mathbf c$ *(díra 1)*

In [ ]:
# TODO 1: sestavte matici A, pravou stranu b a vektor marží c.
#   řádek matice A = jedno omezení (kapacita), sloupec = jeden výkon
#   pořadí řádků: sálové hodiny, lůžkodny
A = ...  # np.array([[...], [...]])  -- tvar 2 x 2
b = ...  # np.array([...])           -- kapacity, délka 2
c = ...  # np.array([...])           -- marže, délka 2

assert A.shape == (2, 2) and b.shape == (2,) and c.shape == (2,), "sedí tvary polí?"
print("A =\n", A, "\nb =", b, "\nc =", c)

## Krok 2 — model v CVXPY *(díra 2)*

Tři řádky, stejné jako u infuze — jen se u jednoho z nich musíte rozhodnout,
kterým směrem se optimalizuje.

In [ ]:
# TODO 2: sestavte úlohu. Pozor na SMĚR -- tady se marže maximalizuje!
x = ...        # cp.Variable(...), oba počty jsou nezáporné
ucel = ...     # cp.Maximize(...) nebo cp.Minimize(...)?
omezeni = ...  # seznam: kapacity se nesmí překročit

In [ ]:
uloha = cp.Problem(ucel, omezeni)
uloha.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)
x_opt = np.asarray(x.value).ravel()

print(f"stav řešení: {uloha.status}")
print(f"x* = ({x_opt[0]:.4f}; {x_opt[1]:.4f}) výkonů za týden")
print(f"marže = {uloha.value:.2f} tis. Kč")

## Krok 3 — kontrola dosazením *(díra 3)*

Dvě nezávislá měřítka. První: řešení se dosadí zpět do kapacit a ověří se, že je
nepřekračuje. Druhé: obě kapacity vyjdou vyčerpané, takže optimum musí ležet
v průsečíku obou omezení — a ten se dá spočítat lineární algebrou bez solveru.

In [ ]:
# TODO 3: dosazením do A spočítejte čerpání obou kapacit.
cerpani = ...
# Ověřte nezápornost počtů a nepřekročení kapacit, s tolerancí 1e-6.
# Pro podmínku platnou pro všechny složky použijte (...).all().
assert ..., "záporný počet výkonů"
assert ..., "řešení překračuje kapacitu"

In [ ]:
for nazev, spotreba, kapacita in zip(["sálové hodiny", "lůžkodny"], cerpani, b):
    print(f"  {nazev:14s} {spotreba:6.2f} / {kapacita:5.1f}")
x_vrchol = np.linalg.solve(A, b)   # průsečík obou omezení, spočítaný bez solveru
print(f"průsečík obou omezení tužkou: ({x_vrchol[0]:.6f}; {x_vrchol[1]:.6f})")
assert np.allclose(x_opt, x_vrchol, atol=1e-6), "optimum neleží v průsečíku omezení"

# kontrola umí spadnout: 18 kyčlí a 0 kolen je taky přípustné, ale marže je nižší
print(f"plán (18; 0): hodiny {A[0] @ [18.0, 0.0]:.0f}, lůžkodny {A[1] @ [18.0, 0.0]:.0f}, "
      f"marže {c @ [18.0, 0.0]:.0f} tis. Kč — přípustné, ale ne optimum")

## Krok 4 — stínové ceny *(díra 4)*

Solver vrátil kromě plánu ještě cenu jednotky kapacity: o kolik by marže
vzrostla, kdyby týden měl o jednu sálovou hodinu (nebo o jeden lůžkoden) víc.

Stínová cena platí v okolí současného zadání. Ověříme ji přepočtem s 41 hodinami.

In [ ]:
# TODO 4: přečtěte dual_value vektorového omezení omezeni[0].
# Pořadí odpovídá řádkům A: hodiny, lůžkodny.
y = ...              # np.asarray(...).ravel()
y_hodiny = ...        # float(...)
y_luzkodny = ...

In [ ]:
print(f"sálová hodina: {y_hodiny:.2f} tis. Kč")
print(f"lůžkoden:      {y_luzkodny:.2f} tis. Kč")
x_navic = cp.Variable(2, nonneg=True)
uloha_navic = cp.Problem(cp.Maximize(c @ x_navic),
                        [A @ x_navic <= b + np.array([1.0, 0.0])])
uloha_navic.solve(solver=cp.CLARABEL)
print(f"marže se 41 hodinami: {uloha_navic.value:.2f} tis. Kč")
assert abs(uloha_navic.value - c @ x_opt - y_hodiny) < 1e-4

## Krok 5 — rozhodnutí

> **Vedení nabízí jednu sálovou hodinu týdně navíc za 8 000 Kč. Vezmete ji?**

Intuice říká „samozřejmě, jedna endoprotéza kyčle nese 62 000 Kč". Odpovězte ale
číslem z duálu, ne intuicí.

### Vaše odpověď

*(dvojklikem do téhle buňky se do ní dá psát; jednu větu, ne výpis)*

**TODO:** Nabídku bereme / nebereme, protože …

## Obrázek (předchystaný)

Přípustná oblast, vrstevnice marže a optimum. Vrstevnice se posouvá od počátku
pryč, dokud se oblasti jen dotýká — a dotkne se jí ve vrcholu.

In [ ]:
xs = np.linspace(0.0, 21.0, 300)
h_hodiny = (b[0] - A[0, 0] * xs) / A[0, 1]     # hranice sálových hodin
h_luzka = (b[1] - A[1, 0] * xs) / A[1, 1]      # hranice lůžkodnů

plt.figure(figsize=(7.6, 5.4))
plt.fill_between(xs, 0.0, np.maximum(np.minimum(h_hodiny, h_luzka), 0.0),
                 color="#dbeafe", label="přípustná oblast")
plt.plot(xs, h_hodiny, color="#7c3aed", lw=2.2, label="sálové hodiny: $2x_1+2{,}5x_2=40$")
plt.plot(xs, h_luzka, color="#b45309", lw=2.2, label="lůžkodny: $5x_1+4x_2=90$")
for m in (400.0, 800.0):
    plt.plot(xs, (m - c[0] * xs) / c[1], color="#6b7280", ls=":", lw=1.1)
plt.plot(xs, (c @ x_opt - c[0] * xs) / c[1], color="#1a7f37", ls="--", lw=1.9,
         label=f"vrstevnice marže {c @ x_opt:.0f} tis. Kč")
plt.plot(x_opt[0], x_opt[1], "o", ms=13, mfc="none", mec="#1a7f37", mew=2.6,
         label=f"optimum ({x_opt[0]:.2f}; {x_opt[1]:.2f})")
plt.xlim(0, 21)
plt.ylim(0, 18.5)
plt.xlabel("endoprotézy kyčle za týden  $x_1$")
plt.ylabel("endoprotézy kolene za týden  $x_2$")
plt.title("Optimum leží ve vrcholu přípustné oblasti")
plt.grid(alpha=0.28)
plt.legend(loc="upper right", fontsize=9.5)
plt.show()

## Bonus: celé počty operací

LP naplánovalo 14,44 kyčle — operovat se ale dají jen celí pacienti. Celočíselnost
se v CVXPY zadá **u proměnné**, parametrem `integer=True`; účel ani omezení se
nemění. Solver pak místo LP řeší celočíselnou úlohu (*mixed-integer LP*, MILP)
metodou větví a mezí. Na dvě proměnné to trvá okamžik; u tisíců proměnných může
být celočíselná úloha výrazně těžší než LP.

Buňka je hotová, stačí ji spustit a pak si s ní hrát.

In [ ]:
x_cele = cp.Variable(2, integer=True)            # jediná změna: celé počty
uloha_cele = cp.Problem(cp.Maximize(c @ x_cele),
                        [A @ x_cele <= b, x_cele >= 0])
uloha_cele.solve()                                # CVXPY vybere solver, který umí MILP
x_int = np.round(x_cele.value) + 0.0          # + 0.0 jen kvůli výpisu „-0“

print(f"LP:               ({x_opt[0]:.2f}; {x_opt[1]:.2f}), marže {c @ x_opt:.0f} tis. Kč")
print(f"celočíselně:      ({x_int[0]:.0f}; {x_int[1]:.0f}), marže {uloha_cele.value:.0f} tis. Kč")
x_dolu = np.floor(x_opt)
print(f"LP zaokrouhlené:  ({x_dolu[0]:.0f}; {x_dolu[1]:.0f}), marže {c @ x_dolu:.0f} tis. Kč")

# Totéž ve SciPy: linprog má parametr integrality, 1 = celočíselná proměnná.
# from scipy.optimize import linprog
# linprog(-c, A_ub=A, b_ub=b, integrality=[1, 1])

**Zkuste si:** Kde leží celočíselné optimum vůči LP optimu a vůči zaokrouhlenému
plánu? Přidejte do `b` jednu sálovou hodinu a porovnejte, o kolik vzroste marže
celočíselného plánu, se stínovou cenou z kroku 4. Platí pořád rozhodnutí o nabídce
za 8 000 Kč?